BIBLIOTECAS

In [ ]:
import pandas as pd
import joblib

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import  f1_score, recall_score, precision_score, accuracy_score
from sklearn.metrics import classification_report

LEITURA DOS ARQUIVOS DESCRITORES DE ETNIAS E LÍNGUAS - CLASSIFICAÇÕES PRÉVIAS E ATUAIS

In [ ]:
caminho_arquivo = 'BDD_etnias_20250505.xlsx';
df_linguas_coleta = pd.read_excel(caminho_arquivo);

caminho_arquivo = 'BDD_linguas_20240604.xlsx';
df_linguas_coleta = pd.read_excel(caminho_arquivo);

caminho_arquivo = 'descritores.xlsx';
df_linguas_novacod = pd.read_excel(caminho_arquivo, sheet_name='etnias');
df_linguas_novacod = pd.read_excel(caminho_arquivo, sheet_name='linguas');

df_ind = pd.read_csv('df_ind_distancia_lingua.csv', sep=';', encoding='utf-8-sig')

DEFINIÇÃO DE VARIÁVEIS NUMÉRICAS E CATEGÓRICAS

In [ ]:
variaveis_numericas = ['VAL_LATITUDE','VAL_LONGITUDE','cod_setor','distancia']
variaveis_categoricas= ['txt_lingua_entrada_coleta','desc_cod_etnia_1_novacod','tipo_setor', 'CD_TI', 'sobrenome','descricao_mais_proxima']

GARANTIR QUE O DATASET DE TREINO TENHA AO MENOS UM REGISTRO DE CADA CLASSIFICAÇÃO RÓTULO

In [ ]:
# Índices com cardinalidade 1
cardinalidade_lingua = df_ind['desc_cod_lingua_1_novacod'].value_counts()
cardinalidade1_lingua = cardinalidade_lingua[cardinalidade_lingua == 1].index

# Registros com cardinalidade 1
df_cardinalidade1 = df_ind[df_ind['desc_cod_lingua_1_novacod'].isin(cardinalidade1_lingua)].copy()

# Registros com cardinalidade diferente de 1
df_cardinalidadedif1 = df_ind[~df_ind['desc_cod_lingua_1_novacod'].isin(cardinalidade1_lingua)].copy()

# Split treino e validacao_teste
X_treino, X_validacao_teste, y_treino, y_validacao_teste = train_test_split(
    df_cardinalidadedif1.drop(columns=['desc_cod_lingua_1_novacod']),  
    df_cardinalidadedif1['desc_cod_lingua_1_novacod'],
    test_size=0.4,
    stratify=df_cardinalidadedif1['desc_cod_lingua_1_novacod']
)

# Inclusão dos registros de cardinalidade 1 no dataset de treino
X_treino = pd.concat([X_treino, df_cardinalidade1.drop(columns=['desc_cod_lingua_1_novacod'])])
y_treino = pd.concat([y_treino, df_cardinalidade1['desc_cod_lingua_1_novacod']])

# Split validacao e teste
X_teste, X_validacao, y_teste, y_validacao = train_test_split(
    X_validacao_teste,  
    y_validacao_teste,
    test_size=0.5,
)

PIPELINES DE PRÉ-PROCESSAMENTO COM ORDINAL E ONE HOT ENCONDERS

In [ ]:
numeric_pipeline = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_pipeline_ordinal = Pipeline(steps=[
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

categorical_pipeline_onehot = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_ordinal = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, variaveis_numericas),
        ('cat', categorical_pipeline_ordinal, variaveis_categoricas)
    ]
)

preprocessor_onehot = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, variaveis_numericas),
        ('cat', categorical_pipeline_onehot, variaveis_categoricas)
    ]
)

display(preprocessor_ordinal)
display(preprocessor_onehot)

PIPILINE DO RANDOM FOREST

In [ ]:
floresta = Pipeline(steps=[
    ('preprocessor', preprocessor_ordinal),
    ('classifier', RandomForestClassifier(
        n_estimators=200,
        min_samples_split=2,
        max_depth=30,
        bootstrap=False
    ))
])

floresta

TREINAMENTO DO MODELO FLORESTA ALEATÓRIA

In [ ]:
floresta.fit(X_treino, y_treino)

GRAVAÇÃO DO MODELO FLORESTA ALEATÓRIA

In [ ]:
joblib.dump(floresta, 'floresta_parametros_lingua.pkl')

PREDIÇÕES DE TREINO FLORESTA ALEATÓRIA

In [ ]:
y_pred_treino = floresta.predict(X_treino)
y_pred_treino

MÉTRICAS DE TREINO FLORESTA ALEATÓRIA

In [ ]:
print(f"Acurácia: {accuracy_score(y_treino, y_pred_treino):.2%}")
print(f"Precisão: {precision_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(f"Revocação: {recall_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(f"F1 Score: {f1_score(y_treino, y_pred_treino, average='macro', zero_division=0):.2%}")
print(classification_report(y_treino, y_pred_treino, zero_division=0))

PREDIÇÕES DE VALIDAÇÃO FLORESTA ALEATÓRIA

In [ ]:
y_pred_validacao = floresta.predict(X_validacao)
y_pred_validacao

MÉTRICAS DE VALIDAÇÃO FLORESTA ALEATÓRIA

In [ ]:
print(f"Acurácia: {accuracy_score(y_validacao, y_pred_validacao):.2%}")
print(f"Precisão: {precision_score(y_validacao, y_pred_validacao, average='macro', zero_division=0):.2%}")
print(f"Revocação: {recall_score(y_validacao, y_pred_validacao, average='macro', zero_division=0):.2%}")
print(f"F1 Score: {f1_score(y_validacao, y_pred_validacao, average='macro', zero_division=0):.2%}")
print(classification_report(y_validacao, y_pred_validacao, zero_division=0))

PREDIÇÕES DE TESTE FLORESTA ALEATÓRIA

In [ ]:
y_pred_teste = floresta.predict(X_teste)
y_pred_teste

MÉTRICAS DE TESTE FLORESTA ALEATÓRIA

In [ ]:
print(f"Acurácia: {accuracy_score(y_teste, y_pred_teste):.2%}")
print(f"Precisão: {precision_score(y_teste, y_pred_teste, average='macro', zero_division=0):.2%}")
print(f"Revocação: {recall_score(y_teste, y_pred_teste, average='macro', zero_division=0):.2%}")
print(f"F1 Score: {f1_score(y_teste, y_pred_teste, average='macro', zero_division=0):.2%}")
print(classification_report(y_teste, y_pred_teste, zero_division=0))